# Private pinned model-cache acquisition on CPU

This version uses only the locked acquisition dependency subset, keeping room for 16.16 GB of pinned Llama/MiniLM cache files on a 20 GB disk. Full research environment installation was verified separately; it must be provisioned separately for GPU smoke. No model load, inference, encoding, training or BigQuery. Cache snapshots are saved as regular files without a duplicate archive. HF_TOKEN stays inside Kaggle Secrets and the download child's stdin.


In [ ]:
from pathlib import Path
input_matches = list(Path("/kaggle/input").rglob("input-bundle.bundle"))
if len(input_matches) != 1:
    raise ValueError("Expected exactly one attached input-bundle.bundle")
BUNDLE_ARCHIVE = str(input_matches[0])
KIT_ROOT = str(input_matches[0].parent)
BUNDLE_SHA256 = "2aa4dac0cb4110783279c3361b4c7618ce601402ed3a40cef168adad75caa6de"
WORKSPACE = "/kaggle/working/b12-input"
RUNTIME = "/kaggle/working/b12-runtime"
BOOTSTRAP = "/kaggle/working/b12-bootstrap"
HF_HOME = "/kaggle/working/hf-home"
ALLOW_INPUT_UNPACK = True
ALLOW_ENV_INSTALL = True
ALLOW_CACHE_DOWNLOADS = True
ALLOW_SECRET_ACCESS = True
CACHE_COMPONENT = "all"


In [ ]:
from kaggle_secrets import UserSecretsClient
try:
    _secret_probe = UserSecretsClient().get_secret("HF_TOKEN")
    if not _secret_probe:
        raise ValueError("empty")
except Exception:
    raise RuntimeError("HF_TOKEN unavailable to this saved run; enable notebook access in Kaggle Secrets") from None
finally:
    if "_secret_probe" in globals():
        del _secret_probe
print("Kaggle secret available; value not displayed or persisted")


In [ ]:
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import subprocess
import sys
import tarfile

EXPECTED_SETUP_FILES = {'scripts/26_kaggle_environment.py': '91be7869ae7223b93e7ff6c3c7a58589b241161ef7a782351f02e105e86d9218', 'configs/kaggle-b12-model-pins.json': '6f72c03fe03c379b1ed0844192ef17c94cb60e908c5269f0e49750a32f04998a', 'configs/kaggle-b12-requirements.lock.txt': 'da3c09c04886fca688aa6a2e54aa1d7c46726be5a4c67b19bc2ba8b9ab36a5cd', 'configs/kaggle-b12-bootstrap.lock.txt': '021d7fd19573db49d3acd74056047912457790f20ceffb4d1196ebffa067a9c8'}
kit = Path(KIT_ROOT)
workspace = Path(WORKSPACE)
runtime_python = str(Path(RUNTIME) / "bin/python")
permitted = ("PATH", "LD_LIBRARY_PATH", "LANG", "LC_ALL")
setup_env = {name: os.environ[name] for name in permitted if name in os.environ}
setup_env.update(UV_CACHE_DIR="/kaggle/working/b12-uv-cache", UV_PYTHON_INSTALL_DIR="/kaggle/working/b12-python", HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_TELEMETRY="1")

def check_setup_files():
    for name, expected in EXPECTED_SETUP_FILES.items():
        target = kit / name
        if target.is_symlink() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError("Setup source fingerprint mismatch: " + name)

if not ALLOW_INPUT_UNPACK:
    print("BLOCKED: input unpack disabled")
else:
    check_setup_files()
    if not Path("/kaggle/working").is_dir():
        raise ValueError("Run this setup in Kaggle")
    archive = Path(BUNDLE_ARCHIVE)
    if hashlib.sha256(archive.read_bytes()).hexdigest() != BUNDLE_SHA256:
        raise ValueError("Bilingual input archive SHA mismatch")
    if workspace.exists():
        raise ValueError("Use a fresh workspace; inputs are immutable")
    with tarfile.open(archive) as bundle:
        entries = bundle.getmembers()
        if len({m.name for m in entries}) != len(entries) or sum(m.size for m in entries) > 128*1024*1024:
            raise ValueError("Archive duplicates/size invalid")
        for item in entries:
            path = PurePosixPath(item.name)
            if not item.isfile() or path.is_absolute() or any(p in {".", ".."} or p.startswith(".") for p in item.name.split("/")):
                raise ValueError("Unsafe archive entry")
        bundle.extractall(workspace, filter="data")
    print("Unpacked accepted input bundle; free disk bytes:", shutil.disk_usage("/kaggle/working").free)


## Small locked acquisition environment

Use only the existing lock dependency closure for huggingface-hub and click. No model or research library version changes. The full 218-package environment is not installed in this cache run.


In [ ]:
if not ALLOW_ENV_INSTALL:
    print("BLOCKED: acquisition environment installation disabled")
else:
    import re
    check_setup_files()
    if not workspace.is_dir():
        raise ValueError("Unpack accepted input bundle first")
    acquisition_names = {
        "certifi", "charset-normalizer", "click", "colorama", "filelock", "fsspec",
        "hf-xet", "huggingface-hub", "idna", "packaging", "pyyaml", "requests",
        "tqdm", "typing-extensions", "urllib3",
    }
    full_lock = (kit / "configs/kaggle-b12-requirements.lock.txt").read_text()
    blocks = re.split(r"(?m)(?=^[A-Za-z0-9])", full_lock)
    selected = []
    found = set()
    for block in blocks:
        match = re.match(r"([A-Za-z0-9_.-]+)==", block)
        if match and match[1] in acquisition_names:
            selected.append(block)
            found.add(match[1])
    acquisition_lock = "".join(selected).encode()
    if found != acquisition_names or hashlib.sha256(acquisition_lock).hexdigest() != "9e619d93825589352748ab8379be6b5e28476b08392eccc8fd4ecf46a6018aa0":
        raise ValueError("Acquisition dependency subset mismatch")
    lock_path = Path("/kaggle/working/b12-acquisition.lock.txt")
    lock_path.write_bytes(acquisition_lock)
    subprocess.run([sys.executable, "-m", "pip", "--isolated", "install", "--no-deps", "--require-hashes", "--target", BOOTSTRAP, "-r", str(kit / "configs/kaggle-b12-bootstrap.lock.txt")], env=setup_env, check=True)
    uv_binary = str(Path(BOOTSTRAP) / "bin/uv")
    if not Path(RUNTIME).exists():
        subprocess.run([uv_binary, "--no-config", "venv", "--python", "3.11.15", "--managed-python", RUNTIME], env=setup_env, check=True)
    subprocess.run([uv_binary, "--no-config", "--no-cache", "pip", "sync", "--python", runtime_python, "--require-hashes", "--index-url", "https://pypi.org/simple", str(lock_path)], env=setup_env, check=True)
    subprocess.run([runtime_python, "--version"], env=setup_env, check=True)
    print("Cache-acquisition environment installed; full inference runtime is a separate stage")


## Pinned model and encoder cache

Retrieve HF_TOKEN only inside Kaggle and pass through child stdin. Download original pinned Llama and MiniLM files; verify exact per-file digests. Authenticated errors are suppressed.


In [ ]:
if not ALLOW_CACHE_DOWNLOADS:
    print("BLOCKED: cache acquisition disabled")
else:
    check_setup_files()
    if CACHE_COMPONENT not in ("all", "encoder", "model"):
        raise ValueError("Invalid cache component")
    pins = json.loads((kit / "configs/kaggle-b12-model-pins.json").read_bytes())
    kinds = ("model", "encoder") if CACHE_COMPONENT == "all" else (CACHE_COMPONENT,)
    expected_bytes = sum(pins["models"][kind]["total_bytes"] for kind in kinds)
    if shutil.disk_usage("/kaggle/working").free < expected_bytes + 2*1024**3:
        raise ValueError("Insufficient free disk for selected cache plus download margin")
    argv = [runtime_python, str(kit / "scripts/26_kaggle_environment.py"), "fetch-cache", "--pins", str(kit / "configs/kaggle-b12-model-pins.json"), "--hf-home", HF_HOME, "--component", CACHE_COMPONENT, "--allow-downloads"]
    cache_env = dict(setup_env, HF_HOME=HF_HOME, HF_HUB_OFFLINE="0", HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_TELEMETRY="1")
    if "model" in kinds:
        if not ALLOW_SECRET_ACCESS:
            raise ValueError("Llama requires explicit secret-access opt-in after approval")
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        try:
            process = subprocess.run([*argv, "--token-stdin"], input=token, text=True, env=cache_env, capture_output=True)
        finally:
            del token
    else:
        process = subprocess.run(argv, text=True, env=cache_env, capture_output=True)
    if process.returncode:
        raise RuntimeError("Cache acquisition blocked/failed; no raw authenticated stderr is displayed")
    cache_evidence = json.loads(process.stdout)
    evidence_path = Path("/kaggle/working/b12-cache-byte-verification.json")
    if evidence_path.exists() and json.loads(evidence_path.read_bytes()) != cache_evidence:
        raise ValueError("Preserve previous cache evidence and choose a fresh evidence path")
    evidence_path.write_text(json.dumps(cache_evidence, sort_keys=True, indent=2) + "\n")
    print(cache_evidence)


## Runtime preflight and handoff

Use the Python 3.11 runtime subprocess for every CLI17/CLI25 call, even if the Kaggle notebook kernel has another version. This cell generates EN/VI configs outside immutable inputs with genuine metadata revisions. Campaign and timeout stay null until the separately authorized smoke/run preparation; runtime remains blocked as intended. Capture notebook URL/version, CPU environment/install logs and cache byte evidence. Stop here: GPU smoke and genuine runs are separate stages. Do not execute the inference notebook unchanged under a different Python kernel.


In [ ]:
if not Path(runtime_python).is_file() or not workspace.is_dir():
    print("BLOCKED: isolated runtime and immutable inputs are required")
else:
    check_setup_files()
    pins = json.loads((kit / "configs/kaggle-b12-model-pins.json").read_bytes())
    for language in ("en", "vi"):
        config = json.loads((workspace / "configs/kaggle-b12-run.template.json").read_bytes())
        config.update(language=language, model_revision=pins["models"]["model"]["revision"], encoder_revision=pins["models"]["encoder"]["revision"], hf_home=HF_HOME)
        path = Path("/kaggle/working") / f"b12-{language}-environment-config.json"
        if path.exists() and json.loads(path.read_bytes()) != config:
            raise ValueError("Preserve previous config; choose a fresh output")
        path.write_text(json.dumps(config, sort_keys=True, indent=2) + "\n")
    print("Pinned config files generated; campaign/time cap remain unresolved, inference disabled")


In [ ]:
# Materialize verified snapshots with hardlinks, keeping only one copy of every payload.
if cache_evidence["status"] != "cache_bytes_verified" or set(cache_evidence["models"]) != {"model", "encoder"}:
    raise ValueError("Both verified caches required before finalization")
inventory_process = subprocess.run(
    [runtime_python, "-c", "import importlib.metadata as m,json,platform; print(json.dumps({'python':platform.python_version(),'purpose':'cache-acquisition-only','packages':sorted((d.metadata['Name'],d.version) for d in m.distributions())}))"],
    env=setup_env, text=True, capture_output=True, check=True,
)
Path("/kaggle/working/b12-installed-acquisition-environment.json").write_text(inventory_process.stdout)
home = Path(HF_HOME)
expected = {}
for kind, model in pins["models"].items():
    repository = home / "hub" / ("models--" + model["repo_id"].replace("/", "--"))
    snapshot = repository / "snapshots" / model["revision"]
    for name, spec in model["files"].items():
        source = snapshot / name
        resolved = source.resolve(strict=True)
        if not resolved.is_relative_to(home.resolve()) or source.stat().st_size != spec["bytes"]:
            raise ValueError("Unsafe or wrong-size snapshot file")
        if source.is_symlink():
            temporary = source.with_name(source.name + ".materialize")
            if temporary.exists() or temporary.is_symlink():
                raise ValueError("Unexpected existing materialization file")
            os.link(resolved, temporary)
            os.replace(temporary, source)
        expected[source.relative_to(home).as_posix()] = spec
    # Every snapshot member is now a regular hardlink; blob names are redundant.
    for redundant in (repository / "blobs", repository / "refs"):
        if redundant.is_symlink():
            raise ValueError("Unexpected cache directory symlink")
        if redundant.exists():
            shutil.rmtree(redundant)
locks = home / "hub" / ".locks"
if locks.exists():
    if locks.is_symlink():
        raise ValueError("Unexpected cache lock directory symlink")
    shutil.rmtree(locks)
actual_files = {p.relative_to(home).as_posix() for p in home.rglob("*") if p.is_file()}
if actual_files != set(expected) or any(p.is_symlink() for p in home.rglob("*")):
    raise ValueError("Final cache contains unpinned files or symlinks")
verified = subprocess.run(
    [runtime_python, str(kit / "scripts/26_kaggle_environment.py"), "verify-cache", "--pins", str(kit / "configs/kaggle-b12-model-pins.json"), "--hf-home", HF_HOME, "--component", "all"],
    env=setup_env, text=True, capture_output=True, check=True,
)
if json.loads(verified.stdout) != cache_evidence:
    raise ValueError("Materialized cache verification mismatch")
manifest = {
    "status": "cache_snapshot_files_verified", "hf_home": HF_HOME,
    "files_verified": len(expected), "payload_bytes": sum(s["bytes"] for s in expected.values()),
    "revisions": {kind: model["revision"] for kind, model in pins["models"].items()},
    "files": expected, "model_loaded": False, "scientific_ready": False,
    "full_research_runtime_retained": False, "separate_full_runtime_install_required": True,
    "storage": "regular snapshot files, one copy per pinned payload, no archive",
}
Path("/kaggle/working/b12-cache-inventory.json").write_text(json.dumps(manifest, sort_keys=True, indent=2) + "\n")
for temporary in (RUNTIME, BOOTSTRAP, "/kaggle/working/b12-uv-cache", "/kaggle/working/b12-python", WORKSPACE):
    target = Path(temporary)
    if target.parent != Path("/kaggle/working") or target.is_symlink():
        raise ValueError("Unexpected temporary setup path")
    if target.exists():
        shutil.rmtree(target)
print({k: v for k, v in manifest.items() if k != "files"})
print("Pinned cache ready for private Kaggle output; GPU smoke/inference have not run")
